# Fashion-MNIST with PySpark MLlib

A port of `TF_Proj2_ZalandoFashion.ipynb` from Keras to Spark MLlib's
`MultilayerPerceptronClassifier`, using the same 784 -> 128 -> 10 architecture.

TensorFlow is used only to download the dataset; all training happens in Spark.

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
jupyter notebook TF_Proj2_ZalandoFashion_PySpark.ipynb
```

Requires a JVM (Java 17 works). Training takes a few minutes on a local CPU.

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StructField, StructType
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.linalg import Vectors, VectorUDT
from tensorflow import keras

spark = (SparkSession.builder
         .appName('fashion-mnist-mlp')
         .master('local[*]')
         .config('spark.driver.memory', '4g')
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')

print(spark.version)

## Loading the data

Each 28x28 image becomes one row holding a 784-element feature vector scaled to 0-1,
the layout MLlib estimators expect.

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

(train_images, train_labels), (test_images, test_labels) = keras.datasets.fashion_mnist.load_data()

SCHEMA = StructType([
    StructField('features', VectorUDT()),
    StructField('label', DoubleType()),
])


def to_spark_df(images, labels):
    flat = (images.reshape(len(images), -1) / 255.0).astype(float)
    pdf = pd.DataFrame({
        'features': [Vectors.dense(row) for row in flat],
        'label': labels.astype(float),
    })
    return spark.createDataFrame(pdf, schema=SCHEMA)


train_df = to_spark_df(train_images, train_labels).cache()
test_df = to_spark_df(test_images, test_labels).cache()

print(train_df.count(), test_df.count())
train_df.printSchema()

In [ ]:
train_df.groupBy('label').count().orderBy('label').show()

## Model

`MultilayerPerceptronClassifier` trains with L-BFGS over the whole dataset rather than
mini-batch Adam, so results differ slightly from the Keras version.

In [ ]:
mlp = MultilayerPerceptronClassifier(
    layers=[784, 128, 10],
    blockSize=128,
    maxIter=100,
    seed=42,
)

model = mlp.fit(train_df)

In [ ]:
predictions = model.transform(test_df)

for metric in ('accuracy', 'f1'):
    score = MulticlassClassificationEvaluator(metricName=metric).evaluate(predictions)
    print(metric, score)

## Per-class accuracy

In [ ]:
per_class = (predictions
             .withColumn('correct', (F.col('prediction') == F.col('label')).cast('double'))
             .groupBy('label')
             .agg(F.count('*').alias('n'), F.avg('correct').alias('accuracy'))
             .orderBy('label')
             .toPandas())
per_class['class'] = [class_names[int(i)] for i in per_class['label']]
print(per_class[['class', 'n', 'accuracy']].to_string(index=False))

## A few predictions

In [ ]:
sample = predictions.select('label', 'prediction').limit(10).toPandas()
for _, row in sample.iterrows():
    print(f"true={class_names[int(row.label)]:<12} predicted={class_names[int(row.prediction)]}")

In [ ]:
spark.stop()